# AI2-THOR：場景多視角與物件座標
依照 [官方 Full Starter Template](https://colab.research.google.com/github/allenai/ai2thor-colab/blob/main/templates/AI2_THOR_Full_Starter_Template.ipynb) 的寫法，使用 ai2thor-colab 啟動 X server、建立 Controller，並用 plot_frames / show_objects_table 顯示結果。

只需上傳這個檔案，選擇 Colab GPU 執行階段，依序執行。
若執行過前一版，先下載需要保留的結果，再「中斷連線並刪除執行階段」後重新連線，避免舊設定影響。

功能：安裝啟動、選擇場景、3／4 個不同位置朝中央拍攝、匯出全部物件與跨視角可見物件、下載 ZIP。


## Installation
沿用官方範本的 Python 套件與 start_xserver() 流程。官方 helper 使用舊版 Xvfb 套件且不等待就緒，因此加入 X11／GLX、24-bit 色深與螢幕尺寸檢查。
若官方啟動失敗或螢幕小於 1920×1920，自動安裝目前系統套件庫的 Xvfb，在空閒 display 啟動 1920×1920 虛擬螢幕並保留日誌。既有 1024×768 螢幕無法直接建立 800×800 Controller。備援使用 Mesa 軟體 OpenGL，可能比 GPU 渲染慢。
[官方 helper 原始碼](https://github.com/allenai/ai2thor-colab/blob/main/ai2thor_colab/__init__.py)


In [ ]:
!pip install --upgrade ai2thor ai2thor-colab


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
import ai2thor
import ai2thor_colab
from ai2thor.controller import Controller
from ai2thor_colab import plot_frames, show_objects_table

# Stop Unity before changing its display connection.
if globals().get('controller') is not None:
    try:
        controller.stop()
    except Exception as exc:
        print('Previous Controller cleanup:', exc)
    finally:
        controller = None

# Query the server extension list: has_extension() also requires client-side support.
# Run the probe in a child process so a broken server cannot hang the notebook.
def x11_ready(display, min_width=1920, min_height=1920):
    env = dict(os.environ, DISPLAY=display)
    code = ("import sys; from Xlib.display import Display; d=Display(); s=d.screen(); "
            "w,h=map(int,sys.argv[1:]); assert 'GLX' in d.list_extensions(), 'X server does not advertise GLX'; "
            "assert s.width_in_pixels >= w and s.height_in_pixels >= h, "
            "f'X11 screen {s.width_in_pixels}x{s.height_in_pixels} is smaller than required {w}x{h}'; "
            "assert s.root_depth == 24, f'X11 screen depth must be 24, got {s.root_depth}'; d.sync(); d.close()")
    try:
        result = subprocess.run([sys.executable, '-c', code, str(min_width), str(min_height)], env=env,
                                capture_output=True, text=True, timeout=4)
        return result.returncode == 0, result.stderr
    except subprocess.TimeoutExpired:
        return False, 'X11 probe timed out'

def wait_for_x11(display, process=None, seconds=20):
    deadline = time.monotonic() + seconds
    error = ''
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            return False, f'Xvfb exited with code {process.returncode}'
        ready, error = x11_ready(display)
        if ready:
            return True, ''
        time.sleep(0.5)
    return False, error

ready, detail = x11_ready(os.environ.get('DISPLAY', ':1'))
if not ready:
    # Official template startup. Its progress bar does not establish readiness.
    try:
        ai2thor_colab.start_xserver()
    except Exception as exc:
        print('Official startup:', exc)
    ready, detail = wait_for_x11(os.environ.get('DISPLAY', ':1'))

if not ready:
    print('X server unavailable or too small; preparing 1920x1920 Xvfb. Last error:', detail)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '--reinstall',
                    'xvfb', 'xauth', 'mesa-utils', 'libgl1-mesa-dri'], check=True)
    # Use a free display; leave unrelated X servers and their lock files alone.
    display_number = next((i for i in range(90, 120)
                           if not Path(f'/tmp/.X{i}-lock').exists()
                           and not Path(f'/tmp/.X11-unix/X{i}').exists()), None)
    if display_number is None:
        raise RuntimeError('No free display in :90..:119')
    display = f':{display_number}'
    x_log = Path('/content/ai2thor_xvfb.log')
    with x_log.open('w') as log:
        _thor_xvfb_process = subprocess.Popen(
            ['Xvfb', display, '-screen', '0', '1920x1920x24', '-ac',
             '+extension', 'GLX', '+render', '-noreset', '-nolisten', 'tcp'],
            stdout=log, stderr=subprocess.STDOUT)
    ready, detail = wait_for_x11(display, _thor_xvfb_process)
    if not ready:
        if _thor_xvfb_process.poll() is None:
            _thor_xvfb_process.terminate()
            _thor_xvfb_process.wait(timeout=5)
        print(x_log.read_text(errors='replace'))
        raise RuntimeError(f'Xvfb startup failed: {detail}. Log: {x_log}')
    os.environ['DISPLAY'] = display
    # Xvfb supplies a virtual screen; Mesa supplies its software OpenGL renderer.
    os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'
    result = subprocess.run(['glxinfo', '-B'], capture_output=True, text=True, timeout=20)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        print(x_log.read_text(errors='replace'))
        raise RuntimeError('X11 is ready but OpenGL initialization failed; provide the output above.')

print('X11 / GLX ready (at least 1920x1920, 24-bit):', os.environ.get('DISPLAY', ':1'))
print('AI2-THOR Version:', ai2thor.__version__)


In [ ]:
ready, detail = x11_ready(os.environ.get('DISPLAY', ':1'))
if not ready:
    raise RuntimeError(f'X server is unavailable or too small; rerun the startup cell. {detail}')

def stop_controller():
    previous = globals().get('controller')
    globals()['controller'] = None
    if previous is not None:
        try:
            previous.stop()
        except Exception as exc:
            print('Previous Controller cleanup:', exc)

def show_thor_logs():
    for log in (Path.home() / '.ai2thor/log/unity.log',
                Path.home() / '.config/unity3d/Allen Institute for Artificial Intelligence/AI2-THOR/Player.log',
                Path('/content/ai2thor_xvfb.log')):
        if log.exists():
            print(f'\nLog: {log}')
            print(log.read_text(errors='replace')[-12000:])

print('Display validated. Unity will be created in the Workspace cell.')


## Workspace：選擇場景
預設 FloorPlan1、4 張 800×800 照片，可改為 3 張或調整 IMAGE_SIZE。HORIZON=0 為水平，正值向下、負值向上。每次重跑此格會關閉舊 Controller，以指定場景、解析度與分割設定建立新 Controller，成功後建立新的輸出資料夾。
server_timeout 在建立 Controller 時設為 300 秒。若仍逾時，會印出 Unity／Xvfb 日誌；請先以 IMAGE_SIZE=600 測試，並提供日誌判斷渲染問題。虛擬螢幕需同時滿足照片的寬度與高度。


In [ ]:
SCENE = 'FloorPlan1' #@param {type:"string"}
NUM_VIEWS = 4 #@param [3, 4] {type:"raw"}
HORIZON = 0 #@param {type:"slider", min:-30, max:60, step:15}
IMAGE_SIZE = 800 #@param [400, 600, 800, 1600] {type:"raw"}

from pathlib import Path
from datetime import datetime, timezone
import copy
import json
import math
import csv
from PIL import Image

if NUM_VIEWS not in (3, 4):
    raise ValueError('NUM_VIEWS must be 3 or 4')
ready, detail = x11_ready(os.environ.get('DISPLAY', ':1'), min_width=IMAGE_SIZE, min_height=IMAGE_SIZE)
if not ready:
    raise RuntimeError(f'Display cannot fit {IMAGE_SIZE}x{IMAGE_SIZE}; rerun the startup cell. {detail}')
stop_controller()
try:
    controller = Controller(scene=SCENE, width=IMAGE_SIZE, height=IMAGE_SIZE,
                            x_display=os.environ.get('DISPLAY', ':1'),
                            gridSize=0.25, snapToGrid=False, renderInstanceSegmentation=True,
                            server_timeout=300.0, server_start_timeout=300.0)
    if not controller.last_event.metadata['lastActionSuccess']:
        raise RuntimeError(controller.last_event.metadata['errorMessage'])
    if controller.last_event.instance_segmentation_frame is None:
        raise RuntimeError('Instance segmentation was not initialized')
except Exception:
    stop_controller()
    show_thor_logs()
    raise

output_dir = Path('/content/ai2thor_outputs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_dir.mkdir(parents=True, exist_ok=False)
print('Output directory:', output_dir)
plot_frames(controller.last_event)


## Show Objects Table：所有物件的座標與朝向
保存全部 metadata objects，包含目前不可見的物件。position XYZ 為世界座標（公尺），Y 向上；rotation XYZ 為 API 回傳的 Euler 角度（度），不代表所有物件都有語意上的正面。
範圍是 AI2-THOR 標註物件，不含未標註的 Unity 幾何。此格先載入全部物件並定義匯出功能，拍攝完成後才寫出檔案。
AI2-THOR 原生 visible 受 visibilityDistance（預設 1.5 公尺）限制，不能用來列出照片中所有可見物件。本 Notebook 改用同一個 event 的 instance_masks：只要物件在任一成功拍攝畫面中至少有 1 個分割像素，匯出的 visible 就為 true，不計入拍攝前視角。
objects.json 保留每個物件最近一次拍攝取得的完整 metadata，原生 visible 另存 ai2thor_visible；visible_views 列出所有可見視角（例如 ['view_1', 'view_3']），visible 等於此清單是否非空。從未可見的物件仍保留，visible_views 為 []。visible_objects.json 從 objects.json 篩選 visible=true，只保留 name、position、rotation、visible_views；座標與角度另存 objects.csv。
判定範圍限於 metadata 中有 objectId 的物件；透明材質的結果依模擬器分割渲染為準。[官方可見性定義](https://ai2thor.allenai.org/ithor/documentation/concepts/)／[instance masks](https://ai2thor.allenai.org/ithor/documentation/environment-state/)。


In [ ]:
event = controller.step(action='ResetObjectFilter')
if not event.metadata['lastActionSuccess']:
    raise RuntimeError(event.metadata['errorMessage'])
objects = copy.deepcopy(event.metadata['objects'])
show_objects_table(objects)

def visible_object_pixel_counts(event):
    if event.instance_segmentation_frame is None or event.instance_masks is None:
        raise RuntimeError('Instance segmentation is unavailable. Rerun the Workspace and Objects cells with renderInstanceSegmentation=True, then capture again.')
    counts = {}
    for obj in event.metadata['objects']:
        object_id = obj['objectId']
        mask = event.instance_masks.get(object_id)
        if mask is not None:
            count = int(mask.sum())
            if count > 0:
                counts[object_id] = count
    return counts

def export_visible_objects(scene_dir: Path) -> Path:
    objects = json.loads((scene_dir / 'objects.json').read_text(encoding='utf-8'))
    visible_objects = [
        {key: obj[key] for key in ('name', 'position', 'rotation', 'visible_views')}
        for obj in objects
        if obj.get('visible') is True
    ]
    output_path = scene_dir / 'visible_objects.json'
    output_path.write_text(
        json.dumps(visible_objects, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    return output_path

print(f'Loaded {len(objects)} objects; exports will be written after capture.')


## 不同位置朝中央拍攝 3／4 張圖片
取得可到達位置，以其 XZ 外接矩形的角落為目標，選擇最近且不重複的位置，面向矩形中央。
真實角落可能被家具占據，因此使用靠近角落的可到達點；視線仍可能被家具遮擋。
傳送失敗會嘗試下一個候選點，圖片使用機器人的第一人稱相機。
每次儲存 view 時，逐一檢查 event.metadata['objects'] 中每個物件的 instance_masks，依 objectId 累積 visible_views。view_1 對應 view_01.png，其餘依此類推。views.json 另存各視角的 view_id 與 visible_object_pixel_counts，供核對。重跑此格會清空 visible_views 並重新累積。


In [ ]:
event = controller.step(action='GetReachablePositions')
if not event.metadata['lastActionSuccess']:
    raise RuntimeError(event.metadata['errorMessage'])
positions = event.metadata['actionReturn']
positions = list({tuple(p[a] for a in 'xyz'): p for p in positions}.values())
if len(positions) < NUM_VIEWS:
    raise RuntimeError('Not enough views; complete capture or choose another scene.')

xmin, xmax = min(p['x'] for p in positions), max(p['x'] for p in positions)
zmin, zmax = min(p['z'] for p in positions), max(p['z'] for p in positions)
center_x, center_z = (xmin + xmax) / 2, (zmin + zmax) / 2
corners = [(xmin, zmin), (xmax, zmin), (xmax, zmax), (xmin, zmax)]
used = set()
views = []
frames = []
# Only saved camera views contribute to the visibility union.
objects_by_id = {obj['objectId']: copy.deepcopy(obj) for obj in objects}
for obj in objects_by_id.values():
    obj.setdefault('ai2thor_visible', obj['visible'])
    obj['visible'] = False
    obj['visible_views'] = []
for index, (corner_x, corner_z) in enumerate(corners[:NUM_VIEWS], 1):
    candidates = sorted(positions, key=lambda p: (p['x'] - corner_x)**2 + (p['z'] - corner_z)**2)
    last_error = ''
    for position in candidates:
        key = tuple(round(position[a], 4) for a in 'xyz')
        if key in used:
            continue
        yaw = math.degrees(math.atan2(center_x - position['x'], center_z - position['z'])) % 360
        event = controller.step(action='Teleport', position=position,
                                rotation=dict(x=0, y=yaw, z=0), horizon=HORIZON, standing=True)
        if not event.metadata['lastActionSuccess']:
            last_error = event.metadata['errorMessage']
            continue
        agent = copy.deepcopy(event.metadata['agent'])
        actual_key = tuple(round(agent['position'][a], 4) for a in 'xyz')
        if actual_key in used:
            continue
        pixel_counts = visible_object_pixel_counts(event)
        used.add(actual_key)
        view_id = f'view_{index}'
        filename = f'view_{index:02d}.png'
        Image.fromarray(event.frame).save(output_dir / filename)
        frames.append(event.frame.copy())
        views.append(dict(view_id=view_id, image=filename, agent=agent,
                          cameraPosition=event.metadata.get('cameraPosition'),
                          fov=event.metadata.get('fov'),
                          visible_object_pixel_counts=pixel_counts))
        for obj in event.metadata['objects']:
            object_id = obj['objectId']
            visible_views = list(objects_by_id.get(object_id, {}).get('visible_views', []))
            if object_id in pixel_counts:
                visible_views.append(view_id)
            objects_by_id[object_id] = copy.deepcopy(obj)
            objects_by_id[object_id]['ai2thor_visible'] = obj['visible']
            objects_by_id[object_id]['visible_views'] = visible_views
            objects_by_id[object_id]['visible'] = bool(visible_views)
        print(filename, 'position:', agent['position'], 'rotation:', agent['rotation'])
        print(f'Image-visible objects: {len(pixel_counts)}; native AI2-THOR visible: {sum(obj["visible"] for obj in event.metadata["objects"])}')
        plot_frames(event)
        break
    else:
        raise RuntimeError(f'Could not capture view {index}: {last_error}')

objects = list(objects_by_id.values())
(output_dir / 'objects.json').write_text(
    json.dumps(objects, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
export_visible_objects(output_dir)
fields = ['objectId', 'objectType', 'name'] + [f'{field}_{axis}' for field in ('position', 'rotation') for axis in 'xyz']
with (output_dir / 'objects.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for obj in objects:
        row = {key: obj.get(key) for key in ('objectId', 'objectType', 'name')}
        for field in ('position', 'rotation'):
            row.update({f'{field}_{axis}': obj[field][axis] for axis in 'xyz'})
        writer.writerow(row)
print(f'Exported {len(objects)} objects, {sum(obj["visible"] for obj in objects)} visible across captured views.')

manifest = dict(scene=SCENE, num_views=len(views), ai2thor_version=ai2thor.__version__,
                position_unit='meter', rotation_unit='degree',
                object_snapshot='latest captured view per object', object_count=len(objects),
                object_visibility='at least one instance segmentation pixel in any captured view',
                visibility_source='instance_masks', min_visible_pixels=1,
                look_at_center_xz=dict(x=center_x, z=center_z), views=views)
(output_dir / 'views.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(output_dir / 'reachable_positions.json').write_text(json.dumps(positions, indent=2), encoding='utf-8')


## 拍攝位置與方向
灰點為可到達位置，箭頭為機器人朝向，紅色叉號為中央。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter([p['x'] for p in positions], [p['z'] for p in positions], s=8, color='lightgray')
ax.scatter(center_x, center_z, marker='x', color='red')
for i, view in enumerate(views, 1):
    p = view['agent']['position']
    yaw = math.radians(view['agent']['rotation']['y'])
    ax.quiver(p['x'], p['z'], math.sin(yaw), math.cos(yaw), angles='xy', scale_units='xy', scale=2)
    ax.annotate(str(i), (p['x'], p['z']))
ax.set(xlabel='X (m)', ylabel='Z (m)', title=SCENE)
ax.set_aspect('equal')
fig.savefig(output_dir / 'viewpoints.png', dpi=150, bbox_inches='tight')
plt.show()


## 下載結果
ZIP 包含照片、完整 objects.json、精簡 visible_objects.json、objects.csv、每張圖的機器人 pose、可到達位置與方向圖。


In [ ]:
import shutil
from google.colab import files

required_files = ('views.json', 'objects.json', 'visible_objects.json', 'objects.csv')
if len(views) != NUM_VIEWS or not all((output_dir / name).exists() for name in required_files):
    raise RuntimeError('Not enough views; complete capture or choose another scene.')
archive = shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir)
files.download(archive)


## 結束
完成或發生錯誤後可執行此格釋放模擬器。


In [ ]:
if globals().get('controller') is not None:
    controller.stop()
    controller = None


## 執行說明與參考
本版保留官方範本流程，補上 X server 就緒檢查及系統 Xvfb 備援。
如果仍啟動失敗，請提供「啟動環境」整格輸出與 show_thor_logs() 的 Unity／Xvfb 日誌。不要只提供 Controller 的最後一段 traceback。若出現螢幕尺寸不足，需重跑啟動環境建立新 Xvfb，重跑 Workspace 不會放大已啟動的虛擬螢幕。
已進行離線語法與模擬拍攝回歸測試，尚未在 Colab GPU／Unity 完整實測。舊輸出沒有 instance masks，需從 Workspace 重新執行並拍攝，才能重建畫面可見物件。

- [官方範本](https://colab.research.google.com/github/allenai/ai2thor-colab/blob/main/templates/AI2_THOR_Full_Starter_Template.ipynb)
- [物件 metadata](https://ai2thor.allenai.org/ithor/documentation/environment-state/)
- [導航](https://ai2thor.allenai.org/ithor/documentation/navigation/)
